# 06 · First look at codes and the association study

This notebook replicates the First look at codes and Association results paragraphs of the report's Results section.

> First look at codes: Door codes appear before 68% of the 3,259 paired tickets vs. 61% of their comparison windows, and electronics codes before 20% vs. 13%.
>
> Association results: The odds that a ticket's own codes appear in its final 7 days rather than in its comparison window's final 7 days are 3.2 times those of other faults' tickets (95% interval 2.3–4.8). By code set, the ratios are rough and do not rank the sets: Main Door is borderline, and Computer Board was not shown to be specific. At 7 days, the pooled result holds in every sensitivity check run so far that gives one, but it rests almost entirely on scrubex machines and is provisional.

We also check the numbers behind its wording, as `docs/08_association_results.md` and the script's tables give them: Main Door's ratio of 2.6 with an interval from 1.01 and a permutation p of 0.08, Computer Board's interval of 0.20–1.34, the pooled ratio's range of 2.33–3.55 over the sensitivity checks, and the 1,402 of 1,439 tickets (97%) on scrubex machines.

**How it works.** `association.py` takes each ticket's final 30 days and a ticket-free 30-day window on the same machine 35 days earlier, with the last 10 distinct non-success codes logged in each, and compares how often a ticket's own code set appears only in its final 7 days vs. only in the comparison window's final 7 days with the same split for other faults' tickets, within machine type and quarter. We rebuild the paired tickets, the first-look shares, and every count, share, and ratio of the association study, including each sensitivity check's pooled ratio, from the per-ticket windows in `fu/fu_ticket_last10.csv` and the per-machine table `fu/fu_devices.csv`. The bootstrap intervals and permutation p-values come from `assoc/as_h2a.csv` and `assoc/as_sensitivities.csv`, and we replay the script's bootstrap and permutations for the primary result with its seed as a cross-check.

**Re-run:** `verification/association.py` (about 2 min). With `RERUN = True`, it rewrites its 17 `assoc/as_*.csv` tables and `assoc/as_summary.json` in a scratch copy of `verification/outputs/`, and the notebook compares each file with the committed one. When IPA's case export is in `data/raw/`, the script also scans its outputs for the export's real identifiers, so the re-created `as_summary.json` lists one more check than the committed copy, which was written without the export, and no table or number changes. `docs/07_association_protocol.md` (the protocol, frozen before any result) and `docs/08_association_results.md` (the results) describe the analysis.

**Used as committed, without re-running:** `association.py` reads only committed files, so the re-run uses them as they are. `fu/fu_ticket_last10.csv` (each ticket's four windows with their last 10 distinct non-success codes) and `fu/fu_devices.csv` (each machine's type, account, and active days) come from `db_followup.py`, and `db/db_codes_buckets.csv` (each code's team bucket) from `verify_with_db.py`; both scripts need the telemetry database, which the notebooks do not re-read. The Category-to-code crosswalk is the frozen `verification/reference/crosswalk_v0.csv`, and one sensitivity check (S5b) also reads `htm_device_exposure.csv` and `fleet/fx_machine_flags.csv`, whose scripts notebooks 02 and 04 re-run. The local text-flag export `flags/flags_by_case.csv` (from `flags.py --write`) is not among the committed outputs, so sensitivity checks S2 and S10 stay pending, as in the report.

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    R.rerun(OUT, 'association.py')
R.compare(OUT)

association.py: exit status 0, 106 s
    Done in 105.5 s: 154 checks passed, 0 failed.
18 files written: 17 identical, 1 DIFFERENT.


,file written by the re-run,compared with the committed file
0,assoc/as_by_model.csv,identical
1,assoc/as_by_type.csv,identical
2,assoc/as_guard.csv,identical
3,assoc/as_h1.csv,identical
4,assoc/as_h2a.csv,identical
5,assoc/as_h2b.csv,identical
6,assoc/as_item2_breakdowns.csv,identical
7,assoc/as_item2_buckets.csv,identical
8,assoc/as_item2_codes.csv,identical
9,assoc/as_item4.csv,identical


## The 3,259 paired tickets

A paired ticket is an unplanned ticket whose 30-day window and comparison window (the 30 days ending 35 days before it) both lie inside the telemetry, with no other ticket on the machine opened from 67 to 5 days before it (docs/07 A2). We apply this rule to the four windows per ticket in `fu/fu_ticket_last10.csv` and check that it picks the same tickets as the script's per-pair table, `assoc/as_pairs.csv`.

In [3]:
import csv
import hashlib
import io
import re

import numpy as np


def read_text(name):
    """A table from the outputs folder with every field as text, as association.py reads it (codes keep their 0s)."""
    return R.read_csv(OUT, name, dtype=str, keep_default_na=False)


def flag(column):
    return column.to_numpy() == 'True'


windows = read_text('fu/fu_ticket_last10.csv')  # one row per ticket and window
win = {w: d.set_index('case_id').sort_index() for w, d in windows.groupby('window')}
T0, C0 = win['corrected_guard0'], win['control_minus35d']  # the 30 days before the ticket; the comparison window
assert all((d.index == T0.index).all() for d in win.values())
ids = T0.index.to_numpy()

unplanned = ~T0.service_group.isin(['Planned work', 'Cancelled request', 'Duplicate case']).to_numpy()
complete = (~flag(T0.lookback_truncated) & ~flag(T0.after_telemetry_end)
            & ~flag(C0.lookback_truncated) & ~flag(C0.after_telemetry_end))
clean = flag(C0.control_clean)  # no other ticket on the machine opened in [A - 67 d, A - 5 d)
paired = unplanned & complete & clean
n_paired = int(paired.sum())

pairs = read_text('assoc/as_pairs.csv').set_index('case_id')  # the script's per-pair table
assert set(ids[paired]) == set(pairs.index[pairs.in_paired == '1']), 'the paired tickets differ from as_pairs.csv'
print(f"{n_paired:,} paired tickets; assoc/as_pairs.csv lists the same {int((pairs.in_paired == '1').sum()):,}.")
pd.Series({'tickets matched to a telemetry device': len(ids), 'unplanned': int(unplanned.sum()),
           'and both windows inside the telemetry': int((unplanned & complete).sum()),
           'and no other ticket from 67 to 5 days before (paired)': n_paired}, name='tickets').to_frame()

3,259 paired tickets; assoc/as_pairs.csv lists the same 3,259.


,tickets
tickets matched to a telemetry device,6113
unplanned,5921
and both windows inside the telemetry,4774
and no other ticket from 67 to 5 days before (paired),3259


## Door codes before 68% of paired tickets vs. 61% of comparison windows, and electronics codes 20% vs. 13%

A code group appears in a window when any code in that team bucket (`db/db_codes_buckets.csv`) is among the window's last 10 distinct non-success codes, which cover its 30 days. We take the shares of paired tickets with Door issue codes and with Electronics/software/comms codes in the ticket window and in the comparison window, and we compare them with the first-look check that `association.py` records in `assoc/as_summary.json`.

In [4]:
code_table = read_text('db/db_codes_buckets.csv')  # each telemetry code's team bucket
bucket_of = dict(zip(code_table.transaction_code, code_table.bucket))
summary = R.read_json(OUT, 'assoc/as_summary.json')
script_check = {c['check']: c['actual'] for c in summary['check_list']}


def has_bucket(window, bucket):
    """Per ticket: a code of the bucket is among the window's last 10 distinct non-success codes."""
    return np.array([any(bucket_of.get(c) == bucket for c in codes.split('|') if c) for codes in window.last10_distinct])


groups = {'Door codes': 'Door issue', 'Electronics codes': 'Electronics/software/comms',
          'Mechanical codes': 'Mechanical fault', 'Stock-out codes': 'Stock-out'}
rows = [(label, has_bucket(T0, b)[paired].mean(), has_bucket(C0, b)[paired].mean(),
         script_check[f'first look: {b} (ticket, control %)']) for label, b in groups.items()]
rows.append(('Any non-success code', (T0.distinct_codes.astype(int) > 0).to_numpy()[paired].mean(),
             (C0.distinct_codes.astype(int) > 0).to_numpy()[paired].mean(),
             script_check['first look: any code (ticket, control %)']))
first_look = pd.DataFrame(rows, columns=['group', 'ticket', 'comparison', 'script']).set_index('group')
door_T, door_C = first_look.ticket['Door codes'], first_look.comparison['Door codes']
elec_T, elec_C = first_look.ticket['Electronics codes'], first_look.comparison['Electronics codes']
print(f'Of {n_paired:,} paired tickets, Door codes appear before {door_T:.1%} vs. {door_C:.1%} of comparison '
      f'windows, and Electronics codes before {elec_T:.1%} vs. {elec_C:.1%}.')
pd.DataFrame({'ticket window, %': (100 * first_look.ticket).round(1),
              'comparison window, %': (100 * first_look.comparison).round(1),
              "association.py's check (as_summary.json)": first_look.script.map(lambda v: f'{v[0]} vs. {v[1]}')})

Of 3,259 paired tickets, Door codes appear before 67.9% vs. 61.1% of comparison windows, and Electronics codes before 20.3% vs. 13.1%.


,"ticket window, %","comparison window, %",association.py's check (as_summary.json)
group,,,
Door codes,67.9,61.1,67.9 vs. 61.1
Electronics codes,20.3,13.1,20.3 vs. 13.1
Mechanical codes,54.6,52.2,54.6 vs. 52.2
Stock-out codes,46.3,45.3,46.3 vs. 45.3
Any non-success code,90.2,89.0,90.2 vs. 89.0


## The association study's pairs, code sets, and machine types

The study drops paired tickets with another ticket in the 5 days before them or with no telemetry on the machine at the time (docs/07 A3), and at 7 days it keeps the pairs whose lists can show whether a code fell in the final 7 days (A8). Each ticket's own code set comes from its Category in the frozen crosswalk `verification/reference/crosswalk_v0.csv`, and a machine type logs a set for a machine when at least 5 other machines of the type show it in a clean comparison window (A10). We rebuild these from `fu/fu_ticket_last10.csv` and `fu/fu_devices.csv` and compare them with `assoc/as_pairs.csv` and the counts pinned in `assoc/as_summary.json`.

In [5]:
h2a = R.read_csv(OUT, 'assoc/as_h2a.csv')  # H2a per code set and pooled, at 7 and 30 days
k7 = float(h2a.loc[h2a.perm_p_role == 'decision', 'k'].item())  # look-back of the deciding row, in hours
K_LIST = 720.0  # 30 days: there the outcome is membership of the last-10 list
codes = sorted(set(code_table.transaction_code) | {c for d in win.values() for s in d.last10_distinct
                                                  for c in s.split('|') if c}, key=lambda c: (len(c), c))
col = {c: j for j, c in enumerate(codes)}


class Window:
    """One window per ticket as arrays: each listed code's hours before the window's end (inf when not listed) and
    its occurrences in the window."""

    def __init__(self, d):
        self.H = np.full((len(d), len(codes)), np.inf)
        self.O = np.zeros((len(d), len(codes)))
        self.distinct = d.distinct_codes.astype(int).to_numpy()
        self.last = np.full(len(d), np.inf)  # hours of the last listed code (the 10th when the list is full)
        rows = zip(d.last10_distinct, d.last10_distinct_hours_before, d.last10_distinct_occurrences)
        for i, (cs, hs, occ) in enumerate(rows):
            if cs:
                hours = [float(x) for x in hs.split('|')]
                for c, hr, oc in zip(cs.split('|'), hours, occ.split('|')):
                    self.H[i, col[c]], self.O[i, col[c]] = hr, float(oc)
                self.last[i] = hours[-1]

    def undetermined(self, k):
        """docs/07 A8: a full list whose 10th code lies within k cannot show whether an unlisted code did too."""
        return (self.distinct > 10) & (self.last < k) if k < K_LIST else np.zeros(len(self.distinct), bool)

    def present(self, k, mask):
        """(tickets x sets): a code of the set is listed with hours below k (k = 720 h: listed at all)."""
        listed = (self.H < k) if k < K_LIST else np.isfinite(self.H)
        return (listed.astype(float) @ mask.astype(float)) > 0


W = {w: Window(d) for w, d in win.items()}
WT, WC = W['corrected_guard0'], W['control_minus35d']

# Machine type (A5), quarter (A6), and account cluster (A11)
devices = read_text('fu/fu_devices.csv').set_index('machine_id')[
    ['machine_type', 'assigned_customer', 'ticket_customer', 'first_active', 'last_active', 'transacting']]
mid = T0.machine_id.to_numpy()
mtype = np.array([re.sub(r'^package name:\s*', '', devices.machine_type.get(m, '').strip().lower()) or 'Unknown'
                  for m in mid], dtype=object)
quarter = pd.to_datetime(T0.opened_at_eastern).dt.to_period('Q').astype(str).to_numpy()
stratum = np.array([f'{t}|{q}' for t, q in zip(mtype, quarter)], dtype=object)
assigned = np.array([devices.assigned_customer.get(m, '') for m in mid], dtype=object)
ticket_customer = np.array([devices.ticket_customer.get(m, '') for m in mid], dtype=object)
account = np.where(assigned != '', assigned, np.where(ticket_customer != '', ticket_customer, mid)).astype(object)

# The frozen crosswalk (docs/07 §3): its hash must equal the one association.py checks
with open(R.VERIFICATION / 'reference' / 'crosswalk_v0.csv', newline='', encoding='utf-8') as f:
    crosswalk = pd.DataFrame(sorted(csv.DictReader(f), key=lambda r: r['category']))
buf = io.StringIO()
csv.writer(buf, lineterminator='\n').writerows(
    [['trouble_code', 'category', 'target_codes', 'target_bucket']]
    + crosswalk[['trouble_code', 'category', 'target_codes', 'target_bucket']].to_numpy().tolist())
assert hashlib.sha256(buf.getvalue().encode('utf-8')).hexdigest() == summary['crosswalk_sha256'], 'crosswalk changed'
cat_set = dict(zip(crosswalk.category, crosswalk.target_codes))
target = np.array([cat_set.get(c, '') if c else '' for c in T0.category], dtype=object)  # each ticket's own set
sets = sorted({s for s in crosswalk.target_codes if s})
set_name = {s: '; '.join(sorted(c for c, x in cat_set.items() if x == s)) for s in sets}


def set_mask(sets_):
    """(codes x sets): the codes of each set."""
    M = np.zeros((len(codes), len(sets_)), bool)
    for j, s in enumerate(sets_):
        M[[col[c] for c in s.split('|')], j] = True
    return M


# Primary population (A3) and the pairs determined at 7 days (A8)
A = pd.to_datetime(T0.anchor_utc).to_numpy()  # ticket creation time, UTC


def other_ticket(lo_days, hi_days):
    """Per ticket: another ticket on the same machine was opened in [A - lo_days, A - hi_days)."""
    out = np.zeros(len(ids), bool)
    for m, g in pd.Series(A, index=np.arange(len(ids))).groupby(mid):
        a = np.sort(g.to_numpy())
        lo = np.searchsorted(a, g.to_numpy() - np.timedelta64(lo_days, 'D'))
        hi = np.searchsorted(a, g.to_numpy() - np.timedelta64(hi_days, 'D'))
        out[g.index.to_numpy()] = hi > lo
    return out


def device_day(column):
    return pd.to_datetime(pd.Series([devices[column].get(m, '') for m in mid]), errors='coerce').to_numpy()


first_active, last_active = device_day('first_active'), device_day('last_active')
transacting = np.array([devices.transacting.get(m, '') == 'True' for m in mid])
window_first_day = (pd.Series(A) - pd.Timedelta(days=30)).dt.normalize().to_numpy()
outside_span = ~transacting | (~pd.isna(last_active) & (last_active < window_first_day))
recent = other_ticket(5, 0)  # another ticket in the 5 days before
primary = paired & ~recent & ~outside_span
det7 = ~(WT.undetermined(k7) | WC.undetermined(k7))
sel7 = primary & det7  # the pairs analysed at every look-back up to 7 days
mapped = target != ''


def able_to_emit(sets_):
    """docs/07 A10 and §14.1, per ticket and set: at least 5 other machines of the ticket's type show a code of the
    set in a clean, complete comparison window. Unknown type never does."""
    ok = flag(C0.control_clean) & ~flag(C0.lookback_truncated)
    shows = pd.DataFrame(WC.present(K_LIST, set_mask(sets_))[ok], columns=sets_).groupby(mid[ok]).any()
    type_of = pd.Series(mtype[ok], index=mid[ok]).groupby(level=0).first()
    machines = shows.groupby(type_of.reindex(shows.index).to_numpy()).sum()  # type x set: machines showing it
    E_ = np.zeros((len(ids), len(sets_)), bool)
    for j, s in enumerate(sets_):
        others = (np.array([machines[s].get(t, 0) for t in mtype])
                  - np.array([bool(shows[s].get(m, False)) for m in mid]))
        E_[:, j] = (mtype != 'Unknown') & (others >= 5)
    return E_


E = able_to_emit(sets)
own = np.array([sets.index(t) if t else -1 for t in target])
h1 = sel7 & (own >= 0) & np.array([E[i, j] if j >= 0 else False for i, j in enumerate(own)])

check_pairs = pairs.loc[ids[paired]]
assert (check_pairs.in_primary.astype(int).to_numpy() == primary[paired]).all()
assert (check_pairs.determined_7d.astype(int).to_numpy() == det7[paired]).all()
assert ((check_pairs.type == mtype[paired]) & (check_pairs.quarter == quarter[paired])
        & (check_pairs.cluster == account[paired]) & (check_pairs.target_codes == target[paired])).all()
pins = summary['pinned_counts']
print(f'Primary look-back: {k7:.0f} hours ({k7 / 24:.0f} days). Crosswalk hash matches as_summary.json; '
      f'population, type, quarter, cluster, and code set agree with as_pairs.csv for every paired ticket.')
pd.DataFrame([
    ('paired tickets (A2)', n_paired, pins['A2 paired tickets']),
    ('primary population (A3)', int(primary.sum()), pins['A3 primary tickets']),
    ('pairs determined at 7 days (A8)', int(sel7.sum()), pins['A8 determined pairs 7d']),
    ('their account clusters (A11)', len(set(account[sel7])), pins['A8 clusters 7d']),
    ('their tickets whose Category has a code set (A7)', int((sel7 & mapped).sum()), pins['A8 mapped 7d']),
    ('of those, on a type that logs the own set (H1, A10)', int(h1.sum()), pins['H1 population 7d']),
], columns=['', 'recomputed', 'pinned in as_summary.json']).set_index('')

Primary look-back: 168 hours (7 days). Crosswalk hash matches as_summary.json; population, type, quarter, cluster, and code set agree with as_pairs.csv for every paired ticket.


,recomputed,pinned in as_summary.json
,,
paired tickets (A2),3259,3259
primary population (A3),2950,2950
pairs determined at 7 days (A8),2911,2911
their account clusters (A11),610,610
their tickets whose Category has a code set (A7),1706,1706
"of those, on a type that logs the own set (H1, A10)",1439,1439


## The pooled specificity ratio at 7 days: 3.2

For a code set, b counts tickets with the set's codes in the final 7 days before the ticket and not in the comparison window's final 7 days, and c counts the reverse. The specificity ratio divides the b-to-c odds of tickets whose Category points at the set by those of other faults' tickets on machine types that log the set, within machine type × quarter (Mantel–Haenszel), and the pooled ratio sums these terms over the sets that pass the entry rule (at least 10 discordant own tickets and non-zero terms). We recompute every count and ratio from the per-pair flags above and compare them with `assoc/as_h2a.csv`, whose deciding row gives the 7-day look-back (168 hours).

In [6]:
def ratio(num, den):
    """docs/07 A13: num/den; +inf when only the denominator is 0; NaN (undefined) when both are."""
    num, den = np.asarray(num, float), np.asarray(den, float)
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(den > 0, num / np.where(den > 0, den, 1), np.where(num > 0, np.inf, np.nan))


class Specificity:
    """docs/07 §4 on the pairs in `sel`. For each code set s: own tickets T (Category points at s) and comparator
    tickets K (Category points at a set sharing no code with s), both on machine types that log s; b = s only in the
    final k before the ticket, c = s only in the comparison window's final k; strata are machine type x quarter."""

    def __init__(self, sel, xt, xc, E_, sets_, target_):
        self.sets = sets_
        codes_of = [set(s.split('|')) for s in sets_]
        self.disjoint = np.array([[not (a & b) for b in codes_of] for a in codes_of])
        self.keys = np.unique(stratum[sel])
        self.h = np.searchsorted(self.keys, stratum[sel])
        self.b, self.c = (xt & ~xc)[sel], (~xt & xc)[sel]
        self.able = E_[sel]  # False on Unknown type
        self.g = np.array([sets_.index(t) if t else -1 for t in target_[sel]])  # own set, -1 when unmapped

    def stat(self, g=None):
        g = self.g if g is None else g
        S = len(self.sets)
        T = (g[:, None] == np.arange(S)) & self.able
        K = (g >= 0)[:, None] & self.disjoint[np.maximum(g, 0)] & self.able
        bT, cT, bK, cK = (np.stack([np.bincount(self.h, weights=x[:, j], minlength=len(self.keys))
                                    for j in range(S)], 1)
                          for x in (self.b & T, self.c & T, self.b & K, self.c & K))  # strata x sets
        n = bT + cT + bK + cK
        with np.errstate(divide='ignore', invalid='ignore'):
            num = np.where(n > 0, bT * cK / np.where(n > 0, n, 1), 0.0).sum(0)
            den = np.where(n > 0, cT * bK / np.where(n > 0, n, 1), 0.0).sum(0)
        enter = ((bT + cT).sum(0) >= 10) & (num > 0) & (den > 0)
        pooled = num[enter].sum() / den[enter].sum() if enter.any() else 0.0  # 0: no set enters
        return dict(T=T, K=K, b_T=bT.sum(0), c_T=cT.sum(0), b_K=bK.sum(0), c_K=cK.sum(0), num=num, den=den,
                    enter=enter, ratio=ratio(num, den), pooled=pooled)


MS = set_mask(sets)
xt7, xc7 = WT.present(k7, MS), WC.present(k7, MS)
spec7 = Specificity(sel7, xt7, xc7, E, sets, target)
obs = spec7.stat()
entering = obs['enter']
pooled = obs['pooled']
h7 = h2a[h2a.k == k7].set_index('set')
counts = pd.DataFrame({'own pairs': obs['T'].sum(0), 'comparator pairs': obs['K'].sum(0), 'own b': obs['b_T'],
                       'own c': obs['c_T'], 'comparator b': obs['b_K'], 'comparator c': obs['c_K'],
                       'enters': entering}, index=sets).astype(int)
counts.loc['POOLED'] = counts[entering].sum()
counts.loc['POOLED', 'enters'] = int(entering.any())
table_cols = ['n_T', 'n_K', 'b_T', 'c_T', 'b_K', 'c_K', 'entered']
assert (counts.to_numpy() == h7.loc[counts.index, table_cols].to_numpy()).all(), 'counts differ from as_h2a.csv'
counts['ratio'] = list(obs['ratio']) + [pooled]
counts['ratio in as_h2a.csv'] = np.exp(h7.loc[counts.index, 'estimate'].astype(float))
assert np.allclose(counts['ratio'], counts['ratio in as_h2a.csv'], equal_nan=True)
crude = (counts.loc['POOLED', 'own b'] / counts.loc['POOLED', 'own c']) / (
    counts.loc['POOLED', 'comparator b'] / counts.loc['POOLED', 'comparator c'])
print(f'Pooled over the {entering.sum()} sets that enter: own tickets {counts.loc["POOLED", "own b"]} vs. '
      f'{counts.loc["POOLED", "own c"]}, other faults\' tickets {counts.loc["POOLED", "comparator b"]} vs. '
      f'{counts.loc["POOLED", "comparator c"]} (crude ratio {crude:.2f}); within machine type and quarter the '
      f'specificity ratio is {pooled:.3f}.')
counts.index = [set_name.get(s, s) for s in counts.index]
counts.round(3)

Pooled over the 5 sets that enter: own tickets 269 vs. 47, other faults' tickets 644 vs. 321 (crude ratio 2.85); within machine type and quarter the specificity ratio is 3.217.


,own pairs,comparator pairs,own b,own c,comparator b,comparator c,enters,ratio,ratio in as_h2a.csv
User Door,218,939,68,12,157,151,1,4.852,4.852
Improper Use,9,398,3,0,43,51,0,NaN,NaN
Computer Board,207,1143,22,11,198,35,1,0.540,0.540
SAM,203,746,89,6,121,67,1,8.016,8.016
Weight Scale,6,184,4,0,19,13,0,inf,inf
Deposit Door,554,376,45,13,45,34,1,4.445,4.445
Main Door,134,815,45,5,123,34,1,2.594,2.594
Counterweight System,0,24,0,0,5,1,0,NaN,NaN
Packer,86,815,33,0,92,58,0,inf,inf
Loading Error,22,903,2,1,74,61,0,0.963,0.963


## Its 95% interval: 2.3–4.8

The interval is the 2.5th to 97.5th percentile of the pooled ratio over the script's 2,000 bootstrap resamples of the 610 account clusters, so we read it from `assoc/as_h2a.csv`. As a cross-check, we redraw the script's cluster weights from its seed, recompute the pooled ratio in each resample over the same entering sets, and count the share of resamples inside the interval, which is its level.

In [7]:
clusters = np.array(sorted(set(account[sel7])), dtype=object)  # the account clusters, sorted (docs/07 §4)
cl = np.searchsorted(clusters, account[sel7])
n_boot, n_cl, S, n_strata = summary['n_boot'], len(clusters), len(sets), len(spec7.keys)
weights = np.random.default_rng(summary['seed']).multinomial(  # fleet_exposure.boot_weights
    n_cl, np.full(n_cl, 1 / n_cl), size=n_boot).astype(float)
cell = spec7.h[:, None] * S + np.arange(S)  # each pair's (stratum, set) cell


def resampled(x):
    """Cluster-weighted sums per (stratum, set) cell in each resample: (resamples x strata x sets)."""
    M = np.zeros((n_cl, n_strata * S))
    np.add.at(M, (np.repeat(cl, S), cell.ravel()), x.astype(float).ravel())
    return (weights @ M).reshape(n_boot, n_strata, S)


def interval(reps):
    """docs/07 A13: 95% percentile interval, inverted_cdf, undefined (NaN) resamples left out."""
    return np.nanpercentile(reps, [2.5, 97.5], method='inverted_cdf')


rbT, rcT, rbK, rcK = (resampled(x) for x in (spec7.b & obs['T'], spec7.c & obs['T'],
                                              spec7.b & obs['K'], spec7.c & obs['K']))
rn = rbT + rcT + rbK + rcK
with np.errstate(divide='ignore', invalid='ignore'):
    rnum = np.where(rn > 0, rbT * rcK / np.where(rn > 0, rn, 1), 0.0).sum(1)
    rden = np.where(rn > 0, rcT * rbK / np.where(rn > 0, rn, 1), 0.0).sum(1)
boot_pooled = ratio(rnum[:, entering].sum(1), rden[:, entering].sum(1))
boot_per_set = ratio(rnum, rden)

lo, hi = np.exp(h7.loc['POOLED', ['ci_lo', 'ci_hi']].astype(float))
replay_lo, replay_hi = interval(boot_pooled)
assert np.allclose([replay_lo, replay_hi], [lo, hi])
inside = (boot_pooled >= replay_lo) & (boot_pooled <= replay_hi)
share_inside = inside.mean()
print(f'{n_boot:,} resamples of {n_cl} account clusters: {int(inside.sum()):,} pooled ratios ({share_inside:.2%}) lie '
      f'inside the interval.')
pd.DataFrame({'pooled ratio': [pooled, pooled], 'interval, low': [lo, replay_lo], 'interval, high': [hi, replay_hi]},
             index=['as_h2a.csv', 'replayed here']).round(4)

2,000 resamples of 610 account clusters: 1,901 pooled ratios (95.05%) lie inside the interval.


,pooled ratio,"interval, low","interval, high"
as_h2a.csv,3.2166,2.343,4.775
replayed here,3.2166,2.343,4.775


## By code set: rough ratios, no ranking, Main Door borderline, and Computer Board not shown to be specific

Each set's ratio comes from the counts above, and its interval and its descriptive permutation p (Category shuffled 2,000 times within machine type × quarter) need the script's resampling, so we read them from `assoc/as_h2a.csv` and replay both with its seed as a cross-check. A set is clear when its interval lies above 1 and its p is at most 0.05, borderline when only one of the two holds, and not shown to be specific when neither does. We call the ratios rough when every set's interval is wider (high end over low end) than the pooled ratio's, and the data rank two sets only when their intervals do not overlap, which we test among the sets with a ratio above 1.

In [8]:
# Replay of the permutations (docs/07 §4 Null, §14.3, §14.14): Category shuffled among the mapped tickets within
# machine type x quarter, strata in (type, quarter) order and tickets in case_id order, from a fresh generator with
# the script's seed; the pooled and per-set p-values use the same draws. In v0 each mapped Category has its own set,
# so shuffling Categories shuffles the own-set labels.
pool, labels = mapped[sel7], stratum[sel7]
order = sorted(set(labels[pool]), key=lambda key: tuple(key.split('|')))
shuffled = [np.flatnonzero(pool & (labels == key)) for key in order]
rng = np.random.default_rng(summary['seed'])
null_set, null_pooled = [], []
for _ in range(summary['n_perm']):
    g = spec7.g.copy()
    for idx in shuffled:
        g[idx] = rng.permutation(spec7.g[idx])
    st = spec7.stat(g)
    with np.errstate(divide='ignore'):
        null_set.append(np.where(st['enter'], np.log(st['ratio']), -np.inf))  # a set that does not enter: -inf
        null_pooled.append(np.log(st['pooled']))
null_set, null_pooled = np.array(null_set), np.array(null_pooled)


def perm_p(observed, null):
    return (1 + np.sum(null >= observed)) / (len(null) + 1)


rows = {}
for j, s in enumerate(sets):
    if not entering[j]:
        continue
    r = h7.loc[s]
    replay_low, replay_high = interval(boot_per_set[:, j])
    rows[set_name[s]] = {
        'ratio': obs['ratio'][j], 'interval, low': np.exp(r.ci_lo), 'interval, high': np.exp(r.ci_hi),
        'permutation p': r.perm_p, 'replayed low': replay_low, 'replayed high': replay_high,
        'replayed p': perm_p(np.log(obs['ratio'][j]), null_set[:, j])}
by_set = pd.DataFrame(rows).T.astype(float)
assert np.allclose(by_set[['interval, low', 'interval, high', 'permutation p']].to_numpy(),
                   by_set[['replayed low', 'replayed high', 'replayed p']].to_numpy())
assert np.isclose(perm_p(np.log(pooled), null_pooled), h7.loc['POOLED', 'perm_p'])


def verdict(low, p):
    above, small = low > 1, p <= 0.05
    return 'clear' if above and small else 'borderline' if above or small else 'not shown to be specific'


by_set['high / low'] = by_set['interval, high'] / by_set['interval, low']
by_set['verdict'] = [verdict(r['interval, low'], r['permutation p']) for _, r in by_set.iterrows()]
pooled_width = hi / lo
rough_label = 'rough' if (by_set['high / low'] > pooled_width).all() else 'as precise as the pooled ratio'
above_one = by_set[by_set.ratio > 1]
rank_label = ('do not rank the sets' if above_one['interval, low'].max() < above_one['interval, high'].min()
              else 'rank some of the sets')
apart = [f'{a} below {b}' for a in by_set.index for b in by_set.index
         if by_set.loc[a, 'interval, high'] < by_set.loc[b, 'interval, low']]
md_row, cb_row = by_set.loc['Main Door'], by_set.loc['Computer Board']
print(f"Each set's interval spans {by_set['high / low'].min():.1f} to {by_set['high / low'].max():.1f} times from low "
      f"to high end vs. {pooled_width:.1f} times for the pooled ratio.")
print(f'Sets above 1 ({", ".join(above_one.index)}): highest low end {above_one["interval, low"].max():.2f} vs. '
      f'lowest high end {above_one["interval, high"].min():.2f}, so their intervals all overlap. Intervals that do '
      f'not overlap: {"; ".join(apart) or "none"}. Pooled permutation p: {perm_p(np.log(pooled), null_pooled):.4f}.')
by_set.drop(columns=['replayed low', 'replayed high']).round(4)

Each set's interval spans 4.9 to 30.5 times from low to high end vs. 2.0 times for the pooled ratio.
Sets above 1 (User Door, SAM, Deposit Door, Main Door): highest low end 3.52 vs. lowest high end 12.50, so their intervals all overlap. Intervals that do not overlap: Computer Board below User Door; Computer Board below SAM; Computer Board below Deposit Door. Pooled permutation p: 0.0005.


,ratio,"interval, low","interval, high",permutation p,replayed p,high / low,verdict
User Door,4.8516,2.5293,12.4995,0.0005,0.0005,4.9418,clear
Computer Board,0.5402,0.2032,1.3448,0.9100,0.9100,6.6186,not shown to be specific
SAM,8.0165,3.5167,34.7640,0.0005,0.0005,9.8853,clear
Deposit Door,4.4454,1.6074,34.3267,0.0015,0.0015,21.3559,clear
Main Door,2.5943,1.0079,30.7136,0.0770,0.0770,30.4714,borderline


## At 7 days, the pooled result holds in every sensitivity check that gives one

Each sensitivity check reruns the pooled test on another population or outcome rule (docs/07 §6); S9 and S11 give no pooled ratio, and S2, S10, and S12 have not been run. We recompute each variant's pooled ratio at 7 days from the per-pair flags and read its interval and permutation p from `assoc/as_sensitivities.csv`. The result holds when the ratio and its interval lie above 1 and p is at most 0.05, and we apply the same rule to the table's 30-day rows.

In [9]:
sens = R.read_csv(OUT, 'assoc/as_sensitivities.csv')
uT, uC = WT.undetermined(k7), WC.undetermined(k7)
lead_in = other_ticket(102, 67)  # another ticket in [A - 102 d, A - 67 d)
history_start = pd.Timestamp('2025-08-01 04:00')  # the case history begins Aug 1st, 2025, 00:00 US Eastern
s4 = primary & ~lead_in & ((pd.Series(A) - pd.Timedelta(days=102)).to_numpy() >= history_start.to_datetime64())
s5_drop = (~pd.isna(first_active) & (first_active > np.datetime64('2025-08-01'))
           & ((pd.Series(A) - pd.Timedelta(days=65)).to_numpy() < first_active + np.timedelta64(90, 'D')))
serial_of = dict(R.read_csv(OUT, 'htm_device_exposure.csv', usecols=['machine_id', 'serial_id']).to_numpy())
install_year = dict(R.read_csv(OUT, 'fleet/fx_machine_flags.csv', usecols=['serial_id', 'install_year']).to_numpy())
year = np.array([install_year.get(serial_of.get(m), np.nan) for m in mid], float)
lifted = complete & clean & ~recent & ~outside_span  # S1: planned, cancelled, and duplicate tickets put back
truncated_ok = (unplanned & ~flag(T0.after_telemetry_end) & ~flag(C0.after_telemetry_end) & clean & ~recent
                & ~outside_span)  # S6: lookback-truncated windows included


def guarded_control(g):
    """S3 (docs/07 §6): the comparison window without codes under g hours, as a lower and an upper bound."""
    present = (((WC.H >= g) & (WC.H < k7)).astype(float) @ MS.astype(float)) > 0
    dropped_repeat = (((WC.H < g) & (WC.O > 1)).astype(float) @ MS.astype(float)) > 0
    unknown = ~present & (dropped_repeat | uC[:, None])
    return present, present | unknown


variants = {'S1 planned/cancelled/duplicate put back': (lifted & det7, xt7, xc7)}
for w, g in (('corrected_guard30', 0.5), ('corrected_guard60', 1.0)):
    low_c, high_c = guarded_control(g)
    keep = primary & det7 & ~W[w].undetermined(k7)
    variants[f'S3 guard {g:g} h, control lower bound'] = (keep, W[w].present(k7, MS), low_c)
    variants[f'S3 guard {g:g} h, control upper bound'] = (keep, W[w].present(k7, MS), high_c)
variants.update({
    'S4 ticket-free lead-in (A >= 11 Nov 2025)': (s4 & det7, xt7, xc7),
    'S5 recently activated machines dropped': (primary & ~s5_drop & det7, xt7, xc7),
    'S5b installed 2025 or later dropped': (primary & ~(year >= 2025) & det7, xt7, xc7),
    'S6 lookback-truncated windows included': (truncated_ok & det7, xt7, xc7),
    'S7 undetermined pairs counted absent': (primary, xt7, xc7),
    'S7 undetermined pairs counted present': (primary, xt7 | uT[:, None], xc7 | uC[:, None]),
    'S8 machine clusters': (sel7, xt7, xc7),
})
recomputed = {name: (int(sel.sum()), Specificity(sel, xt, xc, E, sets, target).stat()['pooled'])
              for name, (sel, xt, xc) in variants.items()}
# S13: Deposit Door's set without its user-behavior codes 19 and 78; A10 recomputed for the new set
sets13 = sorted({'20|79|80' if s == '19|20|78|79|80' else s for s in sets})
target13 = np.array(['20|79|80' if t == '19|20|78|79|80' else t for t in target], dtype=object)
MS13 = set_mask(sets13)
recomputed['S13 Deposit Door without 19 and 78'] = (int(sel7.sum()), Specificity(
    sel7, WT.present(k7, MS13), WC.present(k7, MS13), able_to_emit(sets13), sets13, target13).stat()['pooled'])

rows = sens[sens.h2a.notna() & (sens.variant != 'primary')].copy()
rows[['ratio', 'low', 'high']] = np.exp(rows[['h2a', 'h2a_lo', 'h2a_hi']].astype(float).to_numpy())
rows['holds'] = (rows.ratio > 1) & (rows.low > 1) & (rows.h2a_perm_p <= 0.05)
at7 = rows[rows.k == k7].set_index('variant')
assert set(at7.index) == set(recomputed), (  # every check that gives a pooled ratio is recomputed
    f'as_sensitivities.csv has variants this notebook does not rebuild: {sorted(set(at7.index) - set(recomputed))}')
at7['pairs (recomputed)'] = [recomputed[v][0] for v in at7.index]
at7['pooled ratio (recomputed)'] = [recomputed[v][1] for v in at7.index]
assert (at7['pairs (recomputed)'] == at7.n_pairs).all()
assert np.allclose(at7['pooled ratio (recomputed)'], at7.ratio)
holds_at = {k: bool(d.holds.all()) for k, d in rows.groupby('k')}
days_all_hold = [k / 24 for k, ok in sorted(holds_at.items()) if ok]
n_hold, n_with = int(at7.holds.sum()), len(at7)
low_ratio, high_ratio = at7['pooled ratio (recomputed)'].min(), at7['pooled ratio (recomputed)'].max()
no_ratio = sens[sens.h2a.isna() & sens.k.notna()].variant.unique()
not_run = sens[sens.status.fillna('').str.startswith('pending')]
print(f'At 7 days the pooled result holds in {n_hold} of {n_with} variants (ratio {low_ratio:.2f} to '
      f'{high_ratio:.2f}); every variant holds at {", ".join(f"{d:g}" for d in days_all_hold)} days. At 30 days it '
      f'fails in: {"; ".join(rows[(rows.k == K_LIST) & ~rows.holds].variant)}.')
print(f'No pooled ratio: {"; ".join(no_ratio)}. Not yet run: '
      + '; '.join(f'{v}, {s}' for v, s in zip(not_run.variant, not_run.status)) + '.')
at7[['pairs (recomputed)', 'pooled ratio (recomputed)', 'ratio', 'low', 'high', 'h2a_perm_p', 'holds']].rename(
    columns={'ratio': 'ratio in the table', 'low': 'interval, low', 'high': 'interval, high',
             'h2a_perm_p': 'permutation p'}).rename_axis(None).round(4)

At 7 days the pooled result holds in 13 of 13 variants (ratio 2.33 to 3.55); every variant holds at 7 days. At 30 days it fails in: S4 ticket-free lead-in (A >= 11 Nov 2025).
No pooled ratio: S9 usable unplanned, ticket side; S9 first ticket of each D5 episode; S1 Cancelled request only; S11 time trend: slope x 35 d across control windows. Not yet run: S2 text flags, pending local flag export; S10 interval-based clean rule, pending local flag export; S12 crosswalk v1, pending v1 (IPA code definitions).


,pairs (recomputed),pooled ratio (recomputed),ratio in the table,"interval, low","interval, high",permutation p,holds
S1 planned/cancelled/duplicate put back,3009,3.2059,3.2059,2.2912,4.8773,0.0005,True
"S3 guard 0.5 h, control lower bound",2911,3.3401,3.3401,2.4224,4.9761,0.0005,True
"S3 guard 0.5 h, control upper bound",2911,3.3444,3.3444,2.4237,4.9781,0.0005,True
"S3 guard 1 h, control lower bound",2911,3.1363,3.1363,2.2895,4.6309,0.0005,True
"S3 guard 1 h, control upper bound",2911,3.1558,3.1558,2.2836,4.6978,0.0005,True
S4 ticket-free lead-in (A >= 11 Nov 2025),2297,2.9774,2.9774,2.0534,5.0861,0.0005,True
S5 recently activated machines dropped,2738,2.8450,2.8450,2.0045,4.2586,0.0005,True
S5b installed 2025 or later dropped,2404,2.3298,2.3298,1.6387,3.7098,0.0005,True
S6 lookback-truncated windows included,3711,2.8811,2.8811,2.1072,4.1657,0.0005,True
S7 undetermined pairs counted absent,2950,3.2909,3.2909,2.3512,4.9783,0.0005,True


## Almost entirely scrubex machines: 1,402 of the 1,439 H1 tickets (97%)

The H1 tickets are the 7-day pairs whose Category has a code set that their machine type logs, and docs/08 §7 counts them by machine family to show how far the results rest on scrubex machines. We count them by family (the type's name before its number), next to the pairs at 7 days and the own ticket-set pairs, own discordant pairs, and comparator ticket-set pairs behind the pooled ratio.

In [10]:
family = pd.Series(mtype).str.extract(r'^([a-z]+)', expand=False).fillna('Unknown').to_numpy()
fam7 = family[sel7]
T_in, K_in = obs['T'][:, entering], obs['K'][:, entering]
own_discordant = (spec7.b | spec7.c)[:, entering] & T_in
by_family = pd.DataFrame({f: {'pairs at 7 days': int((fam7 == f).sum()), 'H1 tickets': int((h1 & (family == f)).sum()),
                              'own ticket-set pairs': int(T_in[fam7 == f].sum()),
                              'own discordant pairs': int(own_discordant[fam7 == f].sum()),
                              'comparator ticket-set pairs': int(K_in[fam7 == f].sum())}
                          for f in sorted(set(fam7))}).T.sort_values('pairs at 7 days', ascending=False)
by_family.loc['all'] = by_family.sum()
n_h1, n_scrubex = int(h1.sum()), int(by_family.loc['scrubex', 'H1 tickets'])
share_scrubex = n_scrubex / n_h1
# Machine types with enough H1 tickets to report (docs/07 A12: at least 30 pairs and 10 discordant pairs)
i1 = np.flatnonzero(h1)
x1, y1 = xt7[i1, own[i1]], xc7[i1, own[i1]]
by_type = pd.DataFrame({'type': mtype[i1], 'discordant': x1 != y1}).groupby('type').agg(
    pairs=('discordant', 'size'), discordant=('discordant', 'sum'))
reportable = by_type[(by_type.pairs >= 30) & (by_type.discordant >= 10)].index
type_table = R.read_csv(OUT, 'assoc/as_by_type.csv')  # the script's rows by machine type
type_table = type_table[(type_table.k == k7) & (type_table.set == 'own set')
                        & (type_table.scope == 'emitting types (H1 population)')].set_index('type')
assert (type_table.n_pairs == by_type.pairs.reindex(type_table.index)).all()
assert sorted(type_table.index[type_table.estimable.astype(str) == 'True']) == sorted(reportable)
print(f'{n_scrubex:,} of the {n_h1:,} H1 tickets ({share_scrubex:.1%}) are on scrubex machines, and so are '
      f'{by_family.loc["scrubex", "own discordant pairs"] / by_family.loc["all", "own discordant pairs"]:.1%} of the '
      f'own discordant pairs behind the pooled ratio. The {len(reportable)} machine types with enough H1 tickets to '
      f'report are {", ".join(reportable)}.')
by_family

1,402 of the 1,439 H1 tickets (97.4%) are on scrubex machines, and so are 97.8% of the own discordant pairs behind the pooled ratio. The 6 machine types with enough H1 tickets to report are scrubex-2, scrubex-4, scrubex2.0-2, scrubex2.0-3, scrubex2.0-4, scrubex2.0-5.


,pairs at 7 days,H1 tickets,own ticket-set pairs,own discordant pairs,comparator ticket-set pairs
scrubex,2259,1402,1287,309,3751
alex,372,1,0,0,57
mlc,194,5,0,0,133
sbsx,36,9,7,2,11
sd,34,22,22,5,67
Unknown,14,0,0,0,0
alexsd,2,0,0,0,0
all,2911,1439,1316,316,4019


## Provisional: crosswalk v0 waits for IPA's review

The results use crosswalk v0, which the team approved on Oct 22nd, 2026, and the confirmatory crosswalk v1 waits for IPA's code definitions (sensitivity check S12). We read the crosswalk's version and status columns and the S12 row of `assoc/as_sensitivities.csv`.

In [11]:
s12_status = sens.loc[sens.variant.str.startswith('S12'), 'status'].item()
pending_ipa = crosswalk.status.str.contains('pending IPA').all() and s12_status.startswith('pending')
status_label = 'provisional' if pending_ipa else 'confirmatory'
print(f'Crosswalk versions: {", ".join(crosswalk.version.unique())}; statuses: '
      f'{", ".join(f"{s} ({n} Categories)" for s, n in crosswalk.status.value_counts().items())}. '
      f'S12 (crosswalk v1): {s12_status}.')

Crosswalk versions: v0 (2026-10-22); statuses: team-approved 2026-10-22, pending IPA (41 Categories). S12 (crosswalk v1): pending v1 (IPA code definitions).


## Check against the report

In [12]:
R.check([
    ('Paired tickets', '3,259', R.num(n_paired)),
    ('Door codes among the last 10 codes before paired tickets', '68%', R.pct(door_T)),
    ('Door codes in their comparison windows', '61%', R.pct(door_C)),
    ('Electronics codes before paired tickets', '20%', R.pct(elec_T)),
    ('Electronics codes in their comparison windows', '13%', R.pct(elec_C)),
    ("Final days before the ticket (as_h2a.csv's deciding row)", '7', R.num(k7 / 24)),
    ("Final days of the comparison window (the same look-back)", '7', R.num(k7 / 24)),
    ("Own codes' odds vs. other faults' tickets: pooled specificity ratio", '3.2', R.num(pooled, 1)),
    ('Interval level: share of the bootstrap pooled ratios inside it', '95%', R.pct(share_inside)),
    ('Interval of the pooled ratio', '2.3–4.8', R.span(lo, hi, 1)),
    ("By code set: every set's interval wider than the pooled ratio's", 'rough', rough_label),
    ('By code set: intervals of the sets above 1 all overlap', 'do not rank the sets', rank_label),
    ('Main Door', 'borderline', md_row.verdict),
    ('Main Door: specificity ratio (docs/08)', '2.6', R.num(md_row.ratio, 1)),
    ('Main Door: low end of its interval (docs/08)', '1.01', R.num(md_row['interval, low'], 2)),
    ('Main Door: permutation p (docs/08)', '0.08', R.num(md_row['permutation p'], 2)),
    ('Computer Board', 'not shown to be specific', cb_row.verdict),
    ('Computer Board: interval (as_h2a.csv)', '0.20–1.34', R.span(cb_row['interval, low'], cb_row['interval, high'], 2)),
    ('Look-back at which every sensitivity check holds, days', '7', ', '.join(R.num(d) for d in days_all_hold)),
    ('Sensitivity checks with a pooled ratio in which it holds', 'every',
     'every' if n_hold == n_with else f'{n_hold} of {n_with}'),
    ('Pooled ratio over those checks (docs/08: 2.3 to 3.5)', '2.33–3.55', R.span(low_ratio, high_ratio, 2)),
    ('H1 tickets on scrubex machines (docs/08 §7)', '1,402', R.num(n_scrubex)),
    ('H1 tickets (docs/08 §7)', '1,439', R.num(n_h1)),
    ('Share on scrubex machines (docs/08 figure as_07)', '97%', R.pct(share_scrubex)),
    ('Status of the results (crosswalk v0, pending IPA)', 'provisional', status_label),
]);

,Finding,Report,Recomputed,Match
0,Paired tickets,"3,259","3,259",yes
1,Door codes among the last 10 codes before pair...,68%,68%,yes
2,Door codes in their comparison windows,61%,61%,yes
3,Electronics codes before paired tickets,20%,20%,yes
4,Electronics codes in their comparison windows,13%,13%,yes
5,Final days before the ticket (as_h2a.csv's dec...,7,7,yes
6,Final days of the comparison window (the same ...,7,7,yes
7,Own codes' odds vs. other faults' tickets: poo...,3.2,3.2,yes
8,Interval level: share of the bootstrap pooled ...,95%,95%,yes
9,Interval of the pooled ratio,2.3–4.8,2.3–4.8,yes


All 25 values match the report.
